# 🏆 Final Submission Pipeline - FinanceRAG

## 📋 Overview

Pipeline cuối cùng kết hợp **TẤT CẢ các cải tiến** từ các notebooks trước:

### 🎯 Key Features:

1. **Fine-tuned E5-small Model** 🧠 (from Notebook 5)
   - Fine-tuned trên FinanceRAG data với layer freezing
   - Path: `models/e5-small-financerag-finetuned-v3`

2. **Semantic Chunking** ✨ (from Notebook 3/4)
   - Pre-chunked corpus với optimal settings per dataset
   - Table-aware chunking

3. **Hybrid Retrieval** 🔍 (from Notebook 4)
   - Dense (fine-tuned E5) + BM25
   - Dataset-specific alpha weighting

4. **Advanced Reranking** 📊
   - BAAI/bge-reranker-v2-m3 (SOTA)

5. **Dataset-Specific Optimizations** 🎯 (from Notebook 6/7)
   - MultiHeirTT: top_k=200, 60% BM25
   - TATQA: top_k=150, 50-50 hybrid
   - FinQA/ConvFinQA: top_k=120, slight BM25 boost

6. **E5 Prefixes** 📝
   - Queries: `query: <text>`
   - Documents: `passage: <text>`

---

## 1. Setup & Imports

In [12]:
# Core Libraries
import os
import sys
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from typing import List, Dict, Tuple
from pathlib import Path

# Embedding & Retrieval
from sentence_transformers import SentenceTransformer
import faiss

# Reranking
from FlagEmbedding import FlagReranker

# BM25
from rank_bm25 import BM25Okapi

# PyTorch
import torch

# Utils
import warnings
warnings.filterwarnings('ignore')
import logging
logging.disable(logging.CRITICAL)

# Add parent directory to path
sys.path.insert(0, '..')

# Load configuration
from config import (
    CONFIG, DATASET_SPECIFIC_CONFIG, QRELS_MAPPING,
    E5_QUERY_PREFIX, E5_PASSAGE_PREFIX,
    print_config
)

# Check GPU
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("Running on CPU")
    device = 'cpu'

# Print config
print_config()

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
🏆 FINAL SUBMISSION CONFIGURATION

📊 MODELS:
   ✅ Embedding: ..\..\models\e5-small-financerag-finetuned-v3 (FINE-TUNED)
   Reranker: BAAI/bge-reranker-v2-m3

✂️ CHUNKING:
   ✅ Using PRE-CHUNKED corpus (semantic chunking)
   📂 Source: ..\..\data\chunked_corpus
   Aggregation: max

🔍 RETRIEVAL:
   Hybrid: True (alpha=0.6)
   Top-K: retrieval=100, rerank=50, final=10

🎯 DATASET OVERRIDES:
   multiheirtt: {'top_k_retrieval': 200, 'top_k_rerank': 80, 'hybrid_alpha': 0.4}
   tatqa: {'top_k_retrieval': 150, 'top_k_rerank': 60, 'hybrid_alpha': 0.5}
   finqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   convfinqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   financebench: {'hybrid_alpha': 0.7}
   finder: {'hybrid_alpha': 0.65}
   finqabench: {'hybrid_alpha': 0.6}


## 2. Helper Functions

In [13]:
# ============================================================
# DATA LOADING FUNCTIONS
# ============================================================

def load_jsonl(file_path):
    """Load JSONL file"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def load_jsonl_data(dataset_name: str, data_dir: str):
    """Load corpus, queries, and qrels for a dataset"""
    data_dir = Path(data_dir)
    
    # Load corpus
    corpus_path = data_dir / f'{dataset_name}_corpus.jsonl' / 'corpus.jsonl'
    if not corpus_path.exists():
        corpus_path = data_dir / f'{dataset_name}_corpus_preprocessed.jsonl'
    corpus_df = pd.DataFrame(load_jsonl(corpus_path))
    
    # Load queries
    queries_path = data_dir / f'{dataset_name}_queries.jsonl' / 'queries.jsonl'
    queries_df = pd.DataFrame(load_jsonl(queries_path))
    
    # Load qrels
    qrels_file = QRELS_MAPPING.get(dataset_name, f'{dataset_name}_qrels.tsv')
    qrels_path = data_dir / qrels_file
    qrels_df = pd.read_csv(qrels_path, sep='\t') if qrels_path.exists() else None
    
    print(f"  Loaded: {len(corpus_df)} docs, {len(queries_df)} queries")
    
    return corpus_df, queries_df, qrels_df


def load_prechunked_corpus(dataset_name: str, chunked_dir: str, config_file: str = None):
    """Load pre-chunked corpus from notebook 3/4"""
    chunked_dir = Path(chunked_dir)
    
    # Try optimal file first
    chunked_file = chunked_dir / f"{dataset_name}_corpus_chunked_optimal.jsonl"
    if not chunked_file.exists():
        chunked_file = chunked_dir / f"{dataset_name}_corpus_chunked.jsonl"
    
    if not chunked_file.exists():
        print(f"  ⚠️ Pre-chunked file not found: {chunked_file}")
        return [], None
    
    chunks = load_jsonl(chunked_file)
    
    # Load chunking config
    method = None
    if config_file and Path(config_file).exists():
        with open(config_file, 'r') as f:
            configs = json.load(f)
            if dataset_name in configs:
                method = configs[dataset_name].get('method', 'unknown')
    
    print(f"  ✅ Loaded {len(chunks)} chunks (method: {method})")
    return chunks, method


print("✅ Data loading functions defined")

✅ Data loading functions defined


In [14]:
# ============================================================
# RETRIEVAL FUNCTIONS
# ============================================================

def add_e5_prefix(text: str, is_query: bool = True) -> str:
    """Add E5 prefix to text"""
    prefix = E5_QUERY_PREFIX if is_query else E5_PASSAGE_PREFIX
    return f"{prefix}{text}"


def normalize_scores(scores: np.ndarray) -> np.ndarray:
    """Normalize scores to [0, 1]"""
    min_s, max_s = scores.min(), scores.max()
    if max_s - min_s > 0:
        return (scores - min_s) / (max_s - min_s)
    return np.ones_like(scores)


def hybrid_search(query_emb, query_text, faiss_index, bm25, chunk_texts, top_k, alpha=0.6):
    """Hybrid search: Dense (alpha) + BM25 (1-alpha)"""
    # Dense search
    dense_scores, dense_indices = faiss_index.search(
        query_emb.reshape(1, -1).astype('float32'), 
        min(top_k * 2, faiss_index.ntotal)
    )
    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]
    
    # BM25 search
    query_tokens = query_text.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k * 2]
    
    # Combine candidates
    all_indices = set(dense_indices.tolist()) | set(bm25_top_indices.tolist())
    
    # Normalize and combine scores
    dense_norm = normalize_scores(dense_scores)
    bm25_norm = normalize_scores(bm25_scores[list(all_indices)])
    
    # Create score mapping
    dense_map = {idx: score for idx, score in zip(dense_indices, dense_norm)}
    bm25_map = {idx: bm25_scores[idx] for idx in all_indices}
    bm25_max = max(bm25_map.values()) if bm25_map else 1
    bm25_map = {k: v / bm25_max if bm25_max > 0 else 0 for k, v in bm25_map.items()}
    
    # Combine
    final_scores = []
    for idx in all_indices:
        d_score = dense_map.get(idx, 0)
        b_score = bm25_map.get(idx, 0)
        final_scores.append((idx, alpha * d_score + (1 - alpha) * b_score))
    
    # Sort and return top-k
    final_scores.sort(key=lambda x: x[1], reverse=True)
    top_results = final_scores[:top_k]
    
    return [s for _, s in top_results], [i for i, _ in top_results]


def aggregate_chunk_scores(doc_scores: Dict, method: str = 'max') -> Dict:
    """Aggregate chunk scores to document scores"""
    if method == 'max':
        return {doc_id: max(scores) for doc_id, scores in doc_scores.items()}
    elif method == 'mean':
        return {doc_id: np.mean(scores) for doc_id, scores in doc_scores.items()}
    elif method == 'sum':
        return {doc_id: sum(scores) for doc_id, scores in doc_scores.items()}
    else:
        return {doc_id: max(scores) for doc_id, scores in doc_scores.items()}


print("✅ Retrieval functions defined")

✅ Retrieval functions defined


In [15]:
# ============================================================
# EVALUATION FUNCTIONS
# ============================================================

def compute_ndcg(retrieved: List[str], relevant: List[str], k: int = 10) -> float:
    """Compute NDCG@k for a single query"""
    dcg = 0.0
    for i, doc_id in enumerate(retrieved[:k]):
        if doc_id in relevant:
            dcg += 1.0 / np.log2(i + 2)
    
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_results(results_df: pd.DataFrame, qrels_df: pd.DataFrame, k: int = 10) -> Dict:
    """Evaluate results against qrels"""
    # Detect column names
    query_col = 'query-id' if 'query-id' in qrels_df.columns else 'query_id'
    corpus_col = 'corpus-id' if 'corpus-id' in qrels_df.columns else 'corpus_id'
    
    # Build ground truth
    ground_truth = {}
    for _, row in qrels_df.iterrows():
        qid = str(row[query_col])
        cid = str(row[corpus_col])
        if qid not in ground_truth:
            ground_truth[qid] = []
        ground_truth[qid].append(cid)
    
    # Compute NDCG per query
    ndcg_scores = []
    for qid, group in results_df.groupby('query_id'):
        qid_str = str(qid)
        if qid_str in ground_truth:
            retrieved = group['corpus_id'].astype(str).tolist()[:k]
            relevant = ground_truth[qid_str]
            ndcg = compute_ndcg(retrieved, relevant, k)
            ndcg_scores.append(ndcg)
    
    return {
        'NDCG@10': np.mean(ndcg_scores) if ndcg_scores else 0.0,
        'num_queries': len(ndcg_scores),
        'num_qrels': len(ground_truth)
    }


print("✅ Evaluation functions defined")

✅ Evaluation functions defined


## 3. Load Models

In [16]:
print("Loading models...")

# 1. Embedding model (fine-tuned or base)
print(f"\n1. Loading embedding model: {CONFIG['embedding_model']}")
if CONFIG['use_finetuned']:
    print("   🎯 Using FINE-TUNED model!")
else:
    print("   ⚠️ Fine-tuned model not found, using base model")

embed_model = SentenceTransformer(CONFIG['embedding_model'], device=device)
print("   ✅ Done")

# 2. Reranker model
print(f"\n2. Loading reranker: {CONFIG['reranker_model']}")
reranker = FlagReranker(CONFIG['reranker_model'], use_fp16=(device=='cuda'))
print("   ✅ Done")

print("\n✅ All models loaded!")

Loading models...

1. Loading embedding model: ..\..\models\e5-small-financerag-finetuned-v3
   🎯 Using FINE-TUNED model!
   ✅ Done

2. Loading reranker: BAAI/bge-reranker-v2-m3
   ✅ Done

✅ All models loaded!


## 4. Verify Pre-Chunked Data

In [17]:
# Check pre-chunked files
chunked_dir = Path(CONFIG['chunked_corpus_dir'])
config_file = Path(CONFIG['chunking_config_file'])

print("="*70)
print("🔍 VERIFICATION: Pre-Chunked Data")
print("="*70)

if config_file.exists():
    print(f"\n✅ Config file found: {config_file}")
    
    with open(config_file, 'r') as f:
        chunk_configs = json.load(f)
    
    print(f"\n{'Dataset':<15} {'Method':<25} {'NDCG@10':<12} {'Status':<10}")
    print("-"*70)
    
    available = 0
    for dataset in CONFIG['datasets']:
        if dataset in chunk_configs:
            cfg = chunk_configs[dataset]
            method = cfg.get('method', 'unknown')
            
            if method != 'no_chunking' and cfg.get('chunk_size'):
                method_str = f"{method} ({cfg['chunk_size']}/{cfg.get('chunk_overlap', 0)})"
            else:
                method_str = method
            
            ndcg = cfg.get('ndcg_10', 0.0)
            
            chunked_file = chunked_dir / f"{dataset}_corpus_chunked_optimal.jsonl"
            status = "✅ Ready" if chunked_file.exists() else "❌ Missing"
            if chunked_file.exists():
                available += 1
            
            print(f"{dataset:<15} {method_str:<25} {ndcg:<12.4f} {status:<10}")
        else:
            print(f"{dataset:<15} {'N/A':<25} {'N/A':<12} {'❌ Missing':<10}")
    
    print("-"*70)
    print(f"\n📊 Summary: {available}/{len(CONFIG['datasets'])} datasets have pre-chunked corpus")
else:
    print(f"\n❌ Config file not found: {config_file}")

print("="*70)

🔍 VERIFICATION: Pre-Chunked Data

✅ Config file found: ..\..\data\chunked_corpus\best_chunking_config_per_dataset.json

Dataset         Method                    NDCG@10      Status    
----------------------------------------------------------------------
convfinqa       semantic (2000/0.6)       0.6591       ✅ Ready   
financebench    semantic (1000/0.7)       0.7938       ✅ Ready   
finder          recursive (512/50)        0.7046       ✅ Ready   
finqa           semantic (2500/0.65)      0.6284       ✅ Ready   
finqabench      preserve_tables (1024/100) 0.8676       ✅ Ready   
multiheirtt     semantic (3000/0.65)      0.2131       ✅ Ready   
tatqa           semantic (2000/0.7)       0.5335       ✅ Ready   
----------------------------------------------------------------------

📊 Summary: 7/7 datasets have pre-chunked corpus


## 5. Main Pipeline Function

In [18]:
def process_dataset(dataset_name: str, config: Dict, dataset_overrides: Dict = None):
    """
    Process dataset with:
    1. Fine-tuned E5 model with E5 prefixes
    2. Pre-chunked semantic corpus
    3. Hybrid search (Dense + BM25)
    4. BGE Reranker
    5. Dataset-specific overrides
    """
    print(f"\n{'='*70}")
    print(f"📊 Processing: {dataset_name.upper()}")
    print(f"{'='*70}")
    
    # Apply dataset-specific overrides
    effective_config = config.copy()
    if dataset_overrides and dataset_name in dataset_overrides:
        overrides = dataset_overrides[dataset_name]
        effective_config.update(overrides)
        print(f"🔧 Overrides: {overrides}")
    
    # Load data
    corpus_df, queries_df, qrels_df = load_jsonl_data(dataset_name, effective_config['data_dir'])
    
    # Step 1: Load pre-chunked data
    all_chunks = []
    chunk_to_doc = {}
    
    if effective_config.get('use_prechunked', False):
        print(f"\n📂 Loading pre-chunked corpus...")
        all_chunks, method = load_prechunked_corpus(
            dataset_name,
            effective_config['chunked_corpus_dir'],
            effective_config.get('chunking_config_file')
        )
        
        if all_chunks:
            for c in all_chunks:
                chunk_id = c.get('_id', c.get('chunk_id', ''))
                doc_id = c.get('original_id', c.get('doc_id', chunk_id))
                chunk_to_doc[chunk_id] = doc_id
            print(f"   📈 Expansion: {len(corpus_df)} docs → {len(all_chunks)} chunks")
    
    # Fallback: no chunking
    if not all_chunks:
        print(f"\n⚠️ No pre-chunked data, using original corpus...")
        for _, row in corpus_df.iterrows():
            doc_id = str(row['_id'])
            text = f"{row.get('title', '')} {row.get('text', '')}".strip()
            all_chunks.append({'_id': doc_id, 'text': text, 'original_id': doc_id})
            chunk_to_doc[doc_id] = doc_id
    
    # Extract texts with E5 prefix
    chunk_texts_raw = [c.get('text', '')[:1024] for c in all_chunks]  # Raw for BM25
    chunk_texts = [add_e5_prefix(t, is_query=False) for t in chunk_texts_raw]  # With prefix for embedding
    chunk_ids = [c.get('_id', c.get('chunk_id', '')) for c in all_chunks]
    
    # Step 2: Encode chunks
    print(f"\n🔢 Encoding {len(chunk_texts)} chunks...")
    chunk_embeddings = embed_model.encode(
        chunk_texts,
        batch_size=effective_config['embed_batch_size'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Step 3: Build FAISS index
    print(f"\n🔍 Building FAISS index...")
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings.astype('float32'))
    print(f"   ✅ Index: {index.ntotal} vectors")
    
    # Step 4: Build BM25 index
    bm25 = None
    if effective_config['use_hybrid']:
        print(f"\n🔤 Building BM25 index...")
        tokenized = [t.lower().split() for t in chunk_texts_raw]
        bm25 = BM25Okapi(tokenized)
        print(f"   ⚖️ Alpha: {effective_config['hybrid_alpha']} (Dense) / {1-effective_config['hybrid_alpha']:.1f} (BM25)")
    
    # Free memory
    del chunk_embeddings
    if device == 'cuda':
        torch.cuda.empty_cache()
    
    # Step 5: Process queries
    print(f"\n🎯 Processing {len(queries_df)} queries...")
    query_texts_raw = [str(r.get('text', '')) for _, r in queries_df.iterrows()]
    query_texts = [add_e5_prefix(t, is_query=True) for t in query_texts_raw]  # With E5 prefix
    query_ids = queries_df['_id'].tolist()
    
    query_embeddings = embed_model.encode(
        query_texts,
        batch_size=effective_config['embed_batch_size'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Step 6: Retrieve & Rerank
    print(f"\n🔎 Retrieving & Reranking...")
    print(f"   Config: top_k_retrieval={effective_config['top_k_retrieval']}, top_k_rerank={effective_config['top_k_rerank']}")
    
    results = []
    
    for i, query_id in enumerate(tqdm(query_ids, desc="Retrieve+Rerank")):
        query_emb = query_embeddings[i]
        query_text = query_texts_raw[i]  # Raw text for BM25 and reranking
        
        # Hybrid or dense retrieval
        if effective_config['use_hybrid'] and bm25:
            scores, chunk_indices = hybrid_search(
                query_emb, query_text, index, bm25, chunk_texts_raw,
                effective_config['top_k_retrieval'], effective_config['hybrid_alpha']
            )
        else:
            scores, chunk_indices = index.search(
                query_emb.reshape(1, -1).astype('float32'),
                effective_config['top_k_retrieval']
            )
            scores, chunk_indices = scores[0].tolist(), chunk_indices[0].tolist()
        
        # Aggregate chunks to documents
        doc_scores = {}
        for idx, score in zip(chunk_indices, scores):
            if idx < 0 or idx >= len(chunk_ids):
                continue
            doc_id = chunk_to_doc.get(chunk_ids[idx], chunk_ids[idx])
            if doc_id not in doc_scores:
                doc_scores[doc_id] = []
            doc_scores[doc_id].append(float(score))
        
        doc_agg = aggregate_chunk_scores(doc_scores, effective_config['chunk_aggregation'])
        sorted_docs = sorted(doc_agg.items(), key=lambda x: x[1], reverse=True)[:effective_config['top_k_rerank']]
        
        # Rerank top documents
        candidate_ids = [d[0] for d in sorted_docs]
        candidate_texts = []
        for doc_id in candidate_ids:
            doc_row = corpus_df[corpus_df['_id'] == doc_id]
            if len(doc_row) > 0:
                text = str(doc_row['text'].values[0])[:2048]
            else:
                text = ""
            candidate_texts.append(text)
        
        # Rerank
        pairs = [[query_text, t] for t in candidate_texts]
        rerank_scores = reranker.compute_score(pairs)
        
        if not isinstance(rerank_scores, list):
            rerank_scores = [rerank_scores]
        
        scored = list(zip(candidate_ids, rerank_scores))
        scored.sort(key=lambda x: x[1], reverse=True)
        
        # Store top-k final results
        for doc_id, score in scored[:effective_config['top_k_final']]:
            results.append({
                'query_id': query_id,
                'corpus_id': doc_id,
                'score': float(score)
            })
    
    results_df = pd.DataFrame(results)
    print(f"   ✅ Generated {len(results_df)} results")
    
    # Step 7: Evaluate
    eval_metrics = {}
    if effective_config['eval_on_qrels'] and qrels_df is not None:
        print(f"\n📊 Evaluating...")
        eval_metrics = evaluate_results(results_df, qrels_df)
        print(f"   NDCG@10: {eval_metrics['NDCG@10']:.4f}")
    
    # Clean up
    del query_embeddings, index
    if device == 'cuda':
        torch.cuda.empty_cache()
    
    return results_df, eval_metrics


print("✅ Pipeline function defined")

✅ Pipeline function defined


## 6. Run Complete Pipeline

In [ ]:
all_results = []
all_eval = {}
failed = []

print("\n" + "="*70)
print("🚀 STARTING FINAL PIPELINE")
print("="*70)

print("\n📋 Configuration:")
print(f"   Model: {CONFIG['embedding_model']}")
print(f"   Fine-tuned: {CONFIG['use_finetuned']}")
print(f"   Hybrid: {CONFIG['use_hybrid']}")

print("\n🔧 Dataset-specific overrides:")
for ds in CONFIG['datasets']:
    if ds in DATASET_SPECIFIC_CONFIG:
        print(f"   {ds}: {DATASET_SPECIFIC_CONFIG[ds]}")

for dataset in CONFIG['datasets']:
    try:
        df_res, metrics = process_dataset(
            dataset,
            CONFIG,
            DATASET_SPECIFIC_CONFIG
        )
        all_results.append(df_res)
        if metrics:
            all_eval[dataset] = metrics
    except Exception as e:
        print(f"\n❌ Error processing {dataset}: {e}")
        import traceback
        traceback.print_exc()
        failed.append(dataset)

print(f"\n{'='*70}")
print(f"✅ Pipeline completed: {len(all_results)}/{len(CONFIG['datasets'])} datasets")
if failed:
    print(f"❌ Failed: {failed}")
print(f"{'='*70}")


🚀 STARTING FINAL PIPELINE

📋 Configuration:
   Model: ..\..\models\e5-small-financerag-finetuned-v3
   Fine-tuned: True
   Hybrid: True

🔧 Dataset-specific overrides:
   convfinqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   financebench: {'hybrid_alpha': 0.7}
   finder: {'hybrid_alpha': 0.65}
   finqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   finqabench: {'hybrid_alpha': 0.6}
   multiheirtt: {'top_k_retrieval': 200, 'top_k_rerank': 80, 'hybrid_alpha': 0.4}
   tatqa: {'top_k_retrieval': 150, 'top_k_rerank': 60, 'hybrid_alpha': 0.5}

📊 Processing: CONVFINQA
🔧 Overrides: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
  Loaded: 2066 docs, 421 queries

📂 Loading pre-chunked corpus...
  ✅ Loaded 38909 chunks (method: semantic)
   📈 Expansion: 2066 docs → 38909 chunks

🔢 Encoding 38909 chunks...


Batches:   0%|          | 0/1216 [00:00<?, ?it/s]


🔍 Building FAISS index...
   ✅ Index: 38909 vectors

🔤 Building BM25 index...
   ⚖️ Alpha: 0.55 (Dense) / 0.4 (BM25)

🎯 Processing 421 queries...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]


🔎 Retrieving & Reranking...
   Config: top_k_retrieval=120, top_k_rerank=50


Retrieve+Rerank:   0%|          | 0/421 [00:00<?, ?it/s]

## 7. Evaluation Summary

In [ ]:
if all_eval:
    print("\n" + "="*70)
    print("📊 EVALUATION SUMMARY (NDCG@10)")
    print("="*70)
    
    total_ndcg = 0
    total_queries = 0
    
    for ds, m in sorted(all_eval.items()):
        print(f"\n{ds.upper():15s}: {m['NDCG@10']:.4f} ({m['num_queries']} queries)")
        total_ndcg += m['NDCG@10'] * m['num_qrels']
        total_queries += m['num_qrels']
    
    if total_queries > 0:
        avg_ndcg = total_ndcg / total_queries
        print(f"\n{'='*70}")
        print(f"📈 WEIGHTED AVERAGE NDCG@10: {avg_ndcg:.4f}")
        print(f"{'='*70}")
        
        # Performance tier
        if avg_ndcg >= 0.60:
            print(f"\n🏆 EXCELLENT! Top-tier performance!")
        elif avg_ndcg >= 0.50:
            print(f"\n✅ VERY GOOD! Strong results!")
        elif avg_ndcg >= 0.40:
            print(f"\n✅ GOOD! Solid improvement!")
        else:
            print(f"\n⚠️ More tuning may help")
else:
    print("\n⚠️ No evaluation metrics available")


📊 EVALUATION SUMMARY (NDCG@10)

CONVFINQA      : 0.5091 (126 queries)

FINANCEBENCH   : 0.7347 (45 queries)

FINDER         : 0.3476 (64 queries)

FINQA          : 0.4444 (344 queries)

FINQABENCH     : 0.8759 (30 queries)

MULTIHEIRTT    : 0.1571 (292 queries)

TATQA          : 0.5057 (498 queries)

📈 WEIGHTED AVERAGE NDCG@10: 0.4262

✅ GOOD! Solid improvement!


## 8. Generate Submission

In [ ]:
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    submission_df = final_df[['query_id', 'corpus_id']]
    
    # Save
    submission_df.to_csv(CONFIG['output_file'], index=False)
    
    print(f"\n✅ Submission saved: {CONFIG['output_file']}")
    print(f"   Total entries: {len(submission_df):,}")
    print(f"   Unique queries: {submission_df['query_id'].nunique():,}")
    
    print(f"\n📋 Sample results:")
    print(submission_df.head(15))
    
    # Validation
    counts = submission_df.groupby('query_id').size()
    print(f"\n🔍 Validation:")
    print(f"   Results per query: {dict(counts.value_counts().sort_index())}")
    
    if (counts == 10).all():
        print(f"   ✅ All queries have exactly 10 results")
    else:
        print(f"   ⚠️ Some queries don't have 10 results")
else:
    print("\n❌ No results to save")


✅ Submission saved: ./submission_final.csv
   Total entries: 46,710
   Unique queries: 4,671

📋 Sample results:
     query_id  corpus_id
0   qd4982518  dd4c4f7aa
1   qd4982518  dd4b88920
2   qd4982518  dd4bb016e
3   qd4982518  dd4b9f7f6
4   qd4982518  dd4bb5506
5   qd4982518  dd4bbdb16
6   qd4982518  dd4b87d18
7   qd4982518  dd4bec0ec
8   qd4982518  dd4be45d6
9   qd4982518  dd4bd3790
10  qd49795a8  dd4befb5c
11  qd49795a8  dd4c05bc8
12  qd49795a8  dd4bd7b9c
13  qd49795a8  dd4979602
14  qd49795a8  dd4b9e8ce

🔍 Validation:
   Results per query: {10: 4671}
   ✅ All queries have exactly 10 results


## 9. Final Summary

In [ ]:
print("\n" + "="*70)
print("🏆 FINAL SUBMISSION PIPELINE COMPLETED!")
print("="*70)

print("\n✅ Key Features Applied:")
print("   1. 🧠 Fine-tuned E5-small model (from notebook 5)")
print("   2. ✨ Semantic chunking (from notebook 3/4)")
print("   3. 🔍 Hybrid search (Dense + BM25)")
print("   4. 📊 BGE-reranker-v2-m3 (SOTA)")
print("   5. 📝 E5 prefixes (query: / passage:)")
print("   6. 🎯 Dataset-specific optimizations:")
print("      - MultiHeirTT: top_k=200, BM25-heavy (60%)")
print("      - TATQA: top_k=150, balanced hybrid (50-50)")
print("      - FinQA/ConvFinQA: top_k=120, slight BM25 boost")

if all_eval:
    total_ndcg = sum(m['NDCG@10'] * m['num_qrels'] for m in all_eval.values())
    total_queries = sum(m['num_qrels'] for m in all_eval.values())
    if total_queries > 0:
        avg = total_ndcg / total_queries
        print(f"\n📊 Final NDCG@10: {avg:.4f}")

print(f"\n💾 Output: {CONFIG['output_file']}")
print(f"\n🚀 Ready for Kaggle submission!")
print("="*70)


🏆 FINAL SUBMISSION PIPELINE COMPLETED!

✅ Key Features Applied:
   1. 🧠 Fine-tuned E5-small model (from notebook 5)
   2. ✨ Semantic chunking (from notebook 3/4)
   3. 🔍 Hybrid search (Dense + BM25)
   4. 📊 BGE-reranker-v2-m3 (SOTA)
   5. 📝 E5 prefixes (query: / passage:)
   6. 🎯 Dataset-specific optimizations:
      - MultiHeirTT: top_k=200, BM25-heavy (60%)
      - TATQA: top_k=150, balanced hybrid (50-50)
      - FinQA/ConvFinQA: top_k=120, slight BM25 boost

📊 Final NDCG@10: 0.4262

💾 Output: ./submission_final.csv

🚀 Ready for Kaggle submission!
